# Video Game Market Analysis

**Author:** Amar Gautam
**Tools:** Python (Pandas, Matplotlib, Seaborn), SQL (SQLite)
**Dataset:** Video Game Sales (Kaggle), `vgsales.csv`

## Problem Statement
A new game studio wants to know which genres, platforms, and regions to focus on, based on historical video game sales.

## Business Questions
1. Which genres have the highest global sales?
2. How have total sales changed year by year?
3. Which platforms sell the most?
4. Which publishers dominate the market?
5. Do North America, Europe, and Japan prefer different genres?

> **How to run:** upload `vgsales.csv` in the Colab Files panel, then click **Runtime > Run all**.

## 1. Load the Data

In [ ]:
import os
import pandas as pd

# Try the common places where the file may be (Colab root, /content, local folders)
paths = ["vgsales.csv", "/vgsales.csv", "/content/vgsales.csv", "../data/vgsales.csv", "data/vgsales.csv"]
data_path = next((p for p in paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("vgsales.csv not found. Upload it in the Files panel (folder icon) and run again.")

df = pd.read_csv(data_path)
print("Loaded from:", data_path)
df.head()

## 2. Understand the Data

In [ ]:
print("Rows, columns:", df.shape)
df.info()

In [ ]:
print("Missing values per column:")
df.isnull().sum()

## 3. Clean the Data
Steps: remove duplicates, drop rows with a missing Year or Publisher, fix data types, and remove extra spaces in text columns. I work on a copy so the raw data stays untouched.

In [ ]:
df_clean = df.copy()
original_rows = len(df_clean)

# 1. Duplicates
duplicates = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates()

# 2. Missing Year or Publisher (only a small share of rows, so dropping is safe)
before = len(df_clean)
df_clean = df_clean.dropna(subset=["Year", "Publisher"])
missing_removed = before - len(df_clean)

# 3. Data types: Year from decimal (2006.0) to integer (2006)
df_clean["Year"] = df_clean["Year"].astype(int)

# 4. Remove extra spaces in text columns
for col in ["Name", "Platform", "Genre", "Publisher"]:
    df_clean[col] = df_clean[col].str.strip()

df_clean = df_clean.reset_index(drop=True)

print("Original rows:", original_rows)
print("Duplicates removed:", duplicates)
print("Rows with missing Year/Publisher removed:", missing_removed)
print("Final rows:", len(df_clean))

In [ ]:
# Sanity checks
print("Year range:", df_clean["Year"].min(), "to", df_clean["Year"].max())
print("Rows with sales <= 0:", (df_clean["Global_Sales"] <= 0).sum())
print("Missing values in cleaned data:", int(df_clean.isnull().sum().sum()))

In [ ]:
df_clean.to_csv("vgsales_clean.csv", index=False)
print("Saved vgsales_clean.csv")

### Cleaning Summary
- Original rows: **16,598**
- Duplicates removed: **0**
- Rows with missing Year or Publisher removed: **307** (about 1.8% of the data)
- Year converted from decimal to integer
- Extra spaces removed from text columns
- Final rows: **16,291**

*Why drop instead of fill?* Year and Publisher cannot be guessed reliably, and the removed rows are under 2% of the data, so the results are barely affected.

## 4. Exploratory Analysis

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
os.makedirs("charts", exist_ok=True)   # all charts are saved in this folder

### Question 1: Which genres have the highest global sales?

In [ ]:
genre_sales = (df_clean.groupby("Genre")["Global_Sales"]
               .sum()
               .sort_values(ascending=False))
genre_sales.round(2)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=genre_sales.values, y=genre_sales.index, color="steelblue")
plt.title("Action, Sports and Shooter Lead Global Sales")
plt.xlabel("Global Sales (millions of units)")
plt.ylabel("Genre")
plt.tight_layout()
plt.savefig("charts/q1_genre_sales.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** Action is the top genre with about 1,723M units, followed by Sports (1,309M) and Shooter (1,026M). These three genres together account for roughly 4,060M units.

**Why it matters:** These are the proven high-volume genres, so a new studio has the largest market here. Total sales alone can mislead, though, because Action also has the most games (3,251). See the average sales per game below.

In [ ]:
# Total sales can be high just because a genre has many games. Check sales per game too.
genre_stats = df_clean.groupby("Genre")["Global_Sales"].agg(total_sales="sum", num_games="count", avg_per_game="mean")
genre_stats = genre_stats.sort_values("avg_per_game", ascending=False).round(2)
genre_stats

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=genre_stats["avg_per_game"], y=genre_stats.index, color="mediumpurple")
plt.title("Average Sales per Game by Genre")
plt.xlabel("Average Global Sales per Game (millions of units)")
plt.ylabel("Genre")
plt.tight_layout()
plt.savefig("charts/q1b_avg_sales_per_game.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** Per game, Platform (about 0.95M) and Shooter (about 0.80M) earn the most, ahead of Action (about 0.53M). Adventure and Strategy earn the least per game.

**Why it matters:** Action leads in total because so many Action games exist, not because each one sells best. Shooter and Platform games give a higher return per release, so a studio with a small portfolio may do better there than in the crowded Action market.

### Question 2: How have sales changed year by year?

In [ ]:
yearly = df_clean.groupby("Year").agg(total_sales=("Global_Sales", "sum"), games_released=("Name", "count")).round(2)
print("Peak year:", yearly["total_sales"].idxmax(), "with", yearly["total_sales"].max(), "million units")
yearly.tail(8)

In [ ]:
# The last years in this dataset are incomplete (2017 has only a few games, 2020 only one),
# so the chart stops at 2016.
trend = yearly[yearly.index <= 2016]

plt.figure(figsize=(12, 5))
plt.plot(trend.index, trend["total_sales"], marker="o")
plt.title("Video Game Sales Peaked Around 2008-2009")
plt.xlabel("Year")
plt.ylabel("Global Sales (millions of units)")
plt.tight_layout()
plt.savefig("charts/q2_yearly_sales.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** Sales grew steadily from the mid-1990s and peaked in 2008 (about 679M units), with 2009 almost as high (667M). After 2011 sales fell to about 264M by 2015. The numbers for 2016 and later are incomplete (342 games in 2016, only 3 in 2017, 1 in 2020), so the sharp drop there is partly missing data, not a real crash.

**Why it matters:** Physical-era sales peaked more than ten years ago, so this dataset shows past market structure, not today's. A studio should treat it as a guide to what has sold well, and check recent digital and mobile data before deciding.

### Question 3: Which platforms sell the most?

In [ ]:
platform_sales = (df_clean.groupby("Platform")["Global_Sales"]
                  .sum()
                  .sort_values(ascending=False)
                  .head(10))
platform_sales.round(2)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=platform_sales.values, y=platform_sales.index, color="seagreen")
plt.title("PS2 Is the Best-Selling Platform in the Dataset")
plt.xlabel("Global Sales (millions of units)")
plt.ylabel("Platform")
plt.tight_layout()
plt.savefig("charts/q3_platform_sales.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** PS2 leads with about 1,233M units, followed by Xbox 360 (970M), PS3 (949M), and Wii (910M). Sony, Microsoft, and Nintendo platforms fill the top ten. PS4 appears low (278M) mainly because the dataset ends before its life cycle finished.

**Why it matters:** Console platforms from the big three dominate sales, so a game released on several major consoles reaches the largest audience. Newer platforms are under-counted here, so use recent data before choosing a launch platform.

### Question 4: Which publishers dominate the market?

In [ ]:
publisher_stats = df_clean.groupby("Publisher")["Global_Sales"].agg(total_sales="sum", num_games="count")
publisher_stats["avg_per_game"] = publisher_stats["total_sales"] / publisher_stats["num_games"]
top_publishers = publisher_stats.sort_values("total_sales", ascending=False).head(10).round(2)
top_publishers

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=top_publishers["total_sales"], y=top_publishers.index, color="darkorange")
plt.title("Nintendo Leads Publisher Sales")
plt.xlabel("Global Sales (millions of units)")
plt.ylabel("Publisher")
plt.tight_layout()
plt.savefig("charts/q4_publisher_sales.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** Nintendo leads with about 1,784M units from 696 games, around 2.6M per game. Electronic Arts is second (1,093M) but needed 1,339 games, around 0.8M per game. Activision, Sony Computer Entertainment, and Ubisoft complete the top five.

**Why it matters:** Nintendo wins through strong franchises, not volume, while EA wins through a wide catalog. A new studio cannot match the catalog of large publishers, so it should compete on a few high-quality titles.

### Question 5: Do regions prefer different genres?

In [ ]:
region_genre = df_clean.groupby("Genre")[["NA_Sales", "EU_Sales", "JP_Sales"]].sum().round(2)
region_genre

In [ ]:
region_pct = (region_genre.div(region_genre.sum()) * 100).round(1)
region_pct

In [ ]:
region_pct.plot(kind="bar", figsize=(12, 6))
plt.title("Genre Share of Sales by Region: Japan Prefers Role-Playing")
plt.ylabel("Share of regional sales (%)")
plt.xlabel("Genre")
plt.xticks(rotation=45)
plt.legend(title="Region")
plt.tight_layout()
plt.savefig("charts/q5_region_genre_share.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** North America and Europe look alike: Action, Sports, and Shooter lead in both. Japan is different. Role-Playing is its top genre (about 350M units, 27% of Japan's sales, versus 7.5% in North America), and Shooter makes up only 3% of Japan's sales versus 13% in North America.

**Why it matters:** A shooter or action game fits Western markets, while Japan needs a role-playing focus. A studio should choose its target region before choosing its genre. I used percentage shares because Japan sells fewer games overall, which makes raw totals unfair to compare.

### Extra: Top 10 best-selling games

In [ ]:
df_clean.nlargest(10, "Global_Sales")[["Name", "Platform", "Year", "Genre", "Publisher", "Global_Sales"]].reset_index(drop=True)

## 5. SQL Analysis
The same questions, answered with SQL (SQLite). The results should match the Pandas results above, which confirms the analysis is correct.

In [ ]:
import sqlite3

conn = sqlite3.connect("games.db")
df_clean.to_sql("games", conn, if_exists="replace", index=False)

def run(query):
    return pd.read_sql_query(query, conn)

run("SELECT COUNT(*) AS rows_in_table FROM games;")

In [ ]:
# All SQL queries are stored here once. They are run below and also saved into queries.sql.
queries = {
"Q1: Top genres by global sales": """
SELECT Genre,
       ROUND(SUM(Global_Sales), 2) AS total_sales,
       COUNT(*) AS num_games
FROM games
GROUP BY Genre
ORDER BY total_sales DESC;
""",
"Q2: Sales and games released by year": """
SELECT Year,
       ROUND(SUM(Global_Sales), 2) AS total_sales,
       COUNT(*) AS games_released
FROM games
GROUP BY Year
ORDER BY Year;
""",
"Q3: Top 10 platforms": """
SELECT Platform,
       ROUND(SUM(Global_Sales), 2) AS total_sales
FROM games
GROUP BY Platform
ORDER BY total_sales DESC
LIMIT 10;
""",
"Q4: Top 10 publishers": """
SELECT Publisher,
       ROUND(SUM(Global_Sales), 2) AS total_sales,
       COUNT(*) AS num_games
FROM games
GROUP BY Publisher
ORDER BY total_sales DESC
LIMIT 10;
""",
"Q5: Regional sales by genre": """
SELECT Genre,
       ROUND(SUM(NA_Sales), 2) AS north_america,
       ROUND(SUM(EU_Sales), 2) AS europe,
       ROUND(SUM(JP_Sales), 2) AS japan
FROM games
GROUP BY Genre
ORDER BY north_america DESC;
""",
"Extra 1: Average sales per game by genre (genres with at least 50 games)": """
SELECT Genre,
       COUNT(*) AS num_games,
       ROUND(AVG(Global_Sales), 2) AS avg_sales_per_game
FROM games
GROUP BY Genre
HAVING COUNT(*) >= 50
ORDER BY avg_sales_per_game DESC;
""",
"Extra 2: Top 10 best-selling games": """
SELECT Name, Platform, Year, Genre, Global_Sales
FROM games
ORDER BY Global_Sales DESC
LIMIT 10;
""",
"Extra 3: Best-selling platform in each year (window function)": """
WITH yearly AS (
    SELECT Year, Platform, SUM(Global_Sales) AS sales
    FROM games
    GROUP BY Year, Platform
),
ranked AS (
    SELECT Year, Platform, ROUND(sales, 2) AS sales,
           ROW_NUMBER() OVER (PARTITION BY Year ORDER BY sales DESC) AS rn
    FROM yearly
)
SELECT Year, Platform, sales
FROM ranked
WHERE rn = 1
ORDER BY Year;
""",
}
print(len(queries), "queries ready")

### SQL Q1: Top genres by global sales

In [ ]:
run(queries["Q1: Top genres by global sales"])

### SQL Q2: Sales by year

In [ ]:
run(queries["Q2: Sales and games released by year"])

### SQL Q3: Top 10 platforms

In [ ]:
run(queries["Q3: Top 10 platforms"])

### SQL Q4: Top 10 publishers

In [ ]:
run(queries["Q4: Top 10 publishers"])

### SQL Q5: Regional preferences by genre

In [ ]:
run(queries["Q5: Regional sales by genre"])

**Check:** the SQL totals match the Pandas totals (for example Action = 1,722.84M units, Nintendo = 1,784.43M units), so both methods agree.

### SQL Extra 1: Average sales per game by genre

In [ ]:
run(queries["Extra 1: Average sales per game by genre (genres with at least 50 games)"])

### SQL Extra 2: Top 10 best-selling games

In [ ]:
run(queries["Extra 2: Top 10 best-selling games"])

### SQL Extra 3: Best-selling platform in each year

In [ ]:
run(queries["Extra 3: Best-selling platform in each year (window function)"])

### Save all queries to `queries.sql`

In [ ]:
with open("queries.sql", "w") as f:
    for title, q in queries.items():
        f.write("-- " + title + "\n")
        f.write(q.strip() + "\n\n")
print("Saved queries.sql with", len(queries), "queries")

## 6. Conclusions and Recommendations for the Studio

1. **Genre:** Action, Sports, and Shooter are the biggest markets. Shooter and Platform games earn the most per game, so they suit a studio with a small portfolio.
2. **Platform:** Sony, Microsoft, and Nintendo platforms dominate. Releasing on multiple major consoles reaches the widest audience.
3. **Competition:** Nintendo and Electronic Arts lead, one through strong franchises and the other through a large catalog. A new studio should compete with a few high-quality titles.
4. **Region:** North America and Europe prefer Action, Sports, and Shooter. Japan prefers Role-Playing. Pick the target region first.
5. **Timing:** Sales peaked around 2008-2009 and declined after 2011.

## Limitations
- The data ends around 2016, so recent trends (digital, mobile, live-service games) are missing, and newer platforms such as PS4 are under-counted.
- Sales are units sold, not revenue or profit.
- Years after 2016 are incomplete, so they were left out of the trend chart.
- The dataset has no esports data. Esports earnings could be added as a future extension.

## Next Steps
- Build a Power BI dashboard from `vgsales_clean.csv`.
- Add the Esports Earnings dataset to compare game sales with prize money.